In [2]:
USE modelDB;
GO

Commands completed successfully.

Total execution time: 00:00:00.018

In [3]:
SELECT DB_NAME() AS current_database;

(1 row affected)

Total execution time: 00:00:00.058

current_database
modelDB


In [5]:
IF NOT EXISTS (
    SELECT *
    FROM sysobjects
    WHERE name='bronze_predictions_raw'
    AND xtype='U'
)
BEGIN

    CREATE TABLE bronze_predictions_raw (

        raw_id NVARCHAR(100) PRIMARY KEY,

        received_at NVARCHAR(50) NOT NULL,

        image_filename NVARCHAR(255),

        image_size_kb FLOAT,

        raw_response NVARCHAR(MAX)

    );

END;

Commands completed successfully.

Total execution time: 00:00:00.206

In [6]:
IF NOT EXISTS (
    SELECT *
    FROM sysobjects
    WHERE name='fact_predictions'
    AND xtype='U'
)
BEGIN

    CREATE TABLE fact_predictions (

        prediction_id NVARCHAR(36) PRIMARY KEY,

        timestamp DATETIME2
        NOT NULL
        DEFAULT GETDATE(),

        model_name NVARCHAR(30)
        NOT NULL
        DEFAULT 'cnn',

        result NVARCHAR(20)
        NOT NULL
        CHECK(result IN ('PNEUMONIA','NORMAL')),

        confidence_pct DECIMAL(5,2)
        NOT NULL
        CHECK(confidence_pct BETWEEN 0 AND 100),

        processing_ms INT,

        image_size_kb FLOAT,

        model_version NVARCHAR(20)
        DEFAULT 'v1.0'

    );

END;

Commands completed successfully.

Total execution time: 00:00:00.117

In [7]:
IF NOT EXISTS (
    SELECT *
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME='fact_predictions'
    AND COLUMN_NAME='model_name'
)
BEGIN

    ALTER TABLE fact_predictions
    ADD model_name NVARCHAR(30)
    NOT NULL
    DEFAULT 'cnn';

END;

Commands completed successfully.

Total execution time: 00:00:00.116

In [8]:
CREATE OR ALTER VIEW vw_daily_stats AS

SELECT

CAST(timestamp AS DATE) AS prediction_date,

COUNT(*) AS total_scans,

SUM(
CASE
WHEN result='PNEUMONIA'
THEN 1
ELSE 0
END
) AS pneumonia_cases,

SUM(
CASE
WHEN result='NORMAL'
THEN 1
ELSE 0
END
) AS normal_cases,

ROUND(
100.0 *
SUM(CASE WHEN result='PNEUMONIA' THEN 1 ELSE 0 END)
/ COUNT(*),
2
) AS pneumonia_rate_pct,

ROUND(
AVG(CAST(confidence_pct AS FLOAT)),
2
) AS avg_confidence,

ROUND(
AVG(CAST(processing_ms AS FLOAT)),
0
) AS avg_latency_ms

FROM fact_predictions

GROUP BY CAST(timestamp AS DATE);

Commands completed successfully.

Total execution time: 00:00:00.075

In [9]:
CREATE OR ALTER VIEW vw_summary AS

SELECT

COUNT(*) AS total_scans,

SUM(
CASE
WHEN result='PNEUMONIA'
THEN 1
ELSE 0
END
) AS total_pneumonia,

SUM(
CASE
WHEN result='NORMAL'
THEN 1
ELSE 0
END
) AS total_normal,

ROUND(
AVG(CAST(confidence_pct AS FLOAT)),
2
) AS avg_confidence,

ROUND(
AVG(CAST(processing_ms AS FLOAT)),
0
) AS avg_latency_ms,

MIN(timestamp) AS first_prediction,

MAX(timestamp) AS last_prediction

FROM fact_predictions;

Commands completed successfully.

Total execution time: 00:00:00.033

In [10]:
CREATE OR ALTER VIEW vw_weekly_trend AS

SELECT

FORMAT(timestamp,'yyyy-ww') AS week,

COUNT(*) AS total_scans,

SUM(
CASE
WHEN result='PNEUMONIA'
THEN 1
ELSE 0
END
) AS pneumonia_cases,

ROUND(
AVG(CAST(confidence_pct AS FLOAT)),
2
) AS avg_confidence

FROM fact_predictions

GROUP BY FORMAT(timestamp,'yyyy-ww');

Commands completed successfully.

Total execution time: 00:00:00.039

In [13]:
CREATE OR ALTER VIEW vw_confidence_distribution AS

SELECT

model_name,

CASE

WHEN confidence_pct >= 90
THEN '90-100'

WHEN confidence_pct >= 80
THEN '80-89'

WHEN confidence_pct >= 70
THEN '70-79'

ELSE '<70'

END AS confidence_bucket,

COUNT(*) AS scan_count

FROM fact_predictions

GROUP BY

model_name,

CASE

WHEN confidence_pct >= 90
THEN '90-100'

WHEN confidence_pct >= 80
THEN '80-89'

WHEN confidence_pct >= 70
THEN '70-79'

ELSE '<70'

END;

Commands completed successfully.

Total execution time: 00:00:00.129

In [14]:
CREATE OR ALTER VIEW vw_model_performance AS

SELECT

model_name,

COUNT(*) AS total_scans,

ROUND(
AVG(CAST(confidence_pct AS FLOAT)),
2
) AS avg_confidence_pct,

MIN(confidence_pct) AS min_confidence_pct,

MAX(confidence_pct) AS max_confidence_pct,

ROUND(
AVG(CAST(processing_ms AS FLOAT)),
0
) AS avg_latency_ms,

MIN(processing_ms) AS min_latency_ms,

MAX(processing_ms) AS max_latency_ms

FROM fact_predictions

GROUP BY model_name;

Commands completed successfully.

Total execution time: 00:00:00.018

In [15]:
CREATE OR ALTER VIEW vw_model_summary AS

SELECT

model_name,

COUNT(*) AS total_scans,

ROUND(
AVG(CAST(confidence_pct AS FLOAT)),
2
) AS avg_confidence_pct,

ROUND(
AVG(CAST(processing_ms AS FLOAT)),
0
) AS avg_latency_ms,

ROUND(

AVG(CAST(confidence_pct AS FLOAT))

-

(
AVG(CAST(processing_ms AS FLOAT))
/
100.0
),

2

) AS reliability_score,

MIN(timestamp) AS first_used,

MAX(timestamp) AS last_used

FROM fact_predictions

GROUP BY model_name;

Commands completed successfully.

Total execution time: 00:00:00.038

In [16]:
CREATE OR ALTER VIEW vw_recent_predictions AS

SELECT TOP 20

prediction_id,

FORMAT(
timestamp,
'yyyy-MM-dd HH:mm:ss'
) AS scan_time,

model_name,

result,

confidence_pct,

processing_ms,

image_size_kb

FROM fact_predictions

ORDER BY timestamp DESC;

Commands completed successfully.

Total execution time: 00:00:00.034

In [17]:
SELECT *
FROM INFORMATION_SCHEMA.TABLES;

(9 rows affected)

Total execution time: 00:00:00.104

TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,TABLE_TYPE
modelDB,dbo,bronze_predictions_raw,BASE TABLE
modelDB,dbo,fact_predictions,BASE TABLE
modelDB,dbo,vw_daily_stats,VIEW
modelDB,dbo,vw_summary,VIEW
modelDB,dbo,vw_weekly_trend,VIEW
modelDB,dbo,vw_model_performance,VIEW
modelDB,dbo,vw_confidence_distribution,VIEW
modelDB,dbo,vw_model_summary,VIEW
modelDB,dbo,vw_recent_predictions,VIEW


In [18]:
SELECT *
FROM INFORMATION_SCHEMA.VIEWS;

(7 rows affected)

Total execution time: 00:00:00.067

TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,VIEW_DEFINITION,CHECK_OPTION,IS_UPDATABLE
modelDB,dbo,vw_daily_stats,"CREATE VIEW vw_daily_stats AS SELECT CAST(timestamp AS DATE) AS prediction_date, COUNT(*) AS total_scans, SUM( CASE WHEN result='PNEUMONIA' THEN 1 ELSE 0 END ) AS pneumonia_cases, SUM( CASE WHEN result='NORMAL' THEN 1 ELSE 0 END ) AS normal_cases, ROUND( 100.0 * SUM(CASE WHEN result='PNEUMONIA' THEN 1 ELSE 0 END) / COUNT(*), 2 ) AS pneumonia_rate_pct, ROUND( AVG(CAST(confidence_pct AS FLOAT)), 2 ) AS avg_confidence, ROUND( AVG(CAST(processing_ms AS FLOAT)), 0 ) AS avg_latency_ms FROM fact_predictions GROUP BY CAST(timestamp AS DATE);",NONE,NO
modelDB,dbo,vw_summary,"CREATE VIEW vw_summary AS SELECT COUNT(*) AS total_scans, SUM( CASE WHEN result='PNEUMONIA' THEN 1 ELSE 0 END ) AS total_pneumonia, SUM( CASE WHEN result='NORMAL' THEN 1 ELSE 0 END ) AS total_normal, ROUND( AVG(CAST(confidence_pct AS FLOAT)), 2 ) AS avg_confidence, ROUND( AVG(CAST(processing_ms AS FLOAT)), 0 ) AS avg_latency_ms, MIN(timestamp) AS first_prediction, MAX(timestamp) AS last_prediction FROM fact_predictions;",NONE,NO
modelDB,dbo,vw_weekly_trend,"CREATE VIEW vw_weekly_trend AS SELECT FORMAT(timestamp,'yyyy-ww') AS week, COUNT(*) AS total_scans, SUM( CASE WHEN result='PNEUMONIA' THEN 1 ELSE 0 END ) AS pneumonia_cases, ROUND( AVG(CAST(confidence_pct AS FLOAT)), 2 ) AS avg_confidence FROM fact_predictions GROUP BY FORMAT(timestamp,'yyyy-ww');",NONE,NO
modelDB,dbo,vw_model_performance,"CREATE VIEW vw_model_performance AS SELECT model_name, COUNT(*) AS total_scans, ROUND( AVG(CAST(confidence_pct AS FLOAT)), 2 ) AS avg_confidence_pct, MIN(confidence_pct) AS min_confidence_pct, MAX(confidence_pct) AS max_confidence_pct, ROUND( AVG(CAST(processing_ms AS FLOAT)), 0 ) AS avg_latency_ms, MIN(processing_ms) AS min_latency_ms, MAX(processing_ms) AS max_latency_ms FROM fact_predictions GROUP BY model_name;",NONE,NO
modelDB,dbo,vw_confidence_distribution,"CREATE VIEW vw_confidence_distribution AS SELECT model_name, CASE WHEN confidence_pct >= 90 THEN '90-100' WHEN confidence_pct >= 80 THEN '80-89' WHEN confidence_pct >= 70 THEN '70-79' ELSE '<70' END AS confidence_bucket, COUNT(*) AS scan_count FROM fact_predictions GROUP BY model_name, CASE WHEN confidence_pct >= 90 THEN '90-100' WHEN confidence_pct >= 80 THEN '80-89' WHEN confidence_pct >= 70 THEN '70-79' ELSE '<70' END;",NONE,NO
modelDB,dbo,vw_model_summary,"CREATE VIEW vw_model_summary AS SELECT model_name, COUNT(*) AS total_scans, ROUND( AVG(CAST(confidence_pct AS FLOAT)), 2 ) AS avg_confidence_pct, ROUND( AVG(CAST(processing_ms AS FLOAT)), 0 ) AS avg_latency_ms, ROUND( AVG(CAST(confidence_pct AS FLOAT)) - ( AVG(CAST(processing_ms AS FLOAT)) / 100.0 ), 2 ) AS reliability_score, MIN(timestamp) AS first_used, MAX(timestamp) AS last_used FROM fact_predictions GROUP BY model_name;",NONE,NO
modelDB,dbo,vw_recent_predictions,"CREATE VIEW vw_recent_predictions AS SELECT TOP 20 prediction_id, FORMAT( timestamp, 'yyyy-MM-dd HH:mm:ss' ) AS scan_time, model_name, result, confidence_pct, processing_ms, image_size_kb FROM fact_predictions ORDER BY timestamp DESC;",NONE,NO


In [19]:
INSERT INTO fact_predictions
(
prediction_id,
model_name,
result,
confidence_pct,
processing_ms,
image_size_kb,
model_version
)

VALUES
(
NEWID(),
'cnn',
'PNEUMONIA',
95.50,
230,
512,
'v2.0'
);

(1 row affected)

Total execution time: 00:00:00.138

In [20]:
SELECT *
FROM fact_predictions;

(1 row affected)

Total execution time: 00:00:00.102

prediction_id,timestamp,model_name,result,confidence_pct,processing_ms,image_size_kb,model_version
EB629982-7310-4D2B-BE55-84DFEB2364FD,2026-06-22 17:46:12.9900000,cnn,PNEUMONIA,95.50,230,512,v2.0


In [21]:
SELECT *
FROM vw_model_performance;

(1 row affected)

Total execution time: 00:00:00.094

model_name,total_scans,avg_confidence_pct,min_confidence_pct,max_confidence_pct,avg_latency_ms,min_latency_ms,max_latency_ms
cnn,1,95.5,95.50,95.50,230,230,230


In [22]:
SELECT *
FROM vw_model_summary
ORDER BY reliability_score DESC;

(1 row affected)

Total execution time: 00:00:00.059

model_name,total_scans,avg_confidence_pct,avg_latency_ms,reliability_score,first_used,last_used
cnn,1,95.5,230,93.2,2026-06-22 17:46:12.9900000,2026-06-22 17:46:12.9900000


In [23]:
SELECT *
FROM vw_recent_predictions;

(1 row affected)

Total execution time: 00:00:03.027

prediction_id,scan_time,model_name,result,confidence_pct,processing_ms,image_size_kb
EB629982-7310-4D2B-BE55-84DFEB2364FD,2026-06-22 17:46:12,cnn,PNEUMONIA,95.50,230,512


In [24]:
SELECT *
FROM vw_summary;

(1 row affected)

Total execution time: 00:00:00.057

total_scans,total_pneumonia,total_normal,avg_confidence,avg_latency_ms,first_prediction,last_prediction
1,1,0,95.5,230,2026-06-22 17:46:12.9900000,2026-06-22 17:46:12.9900000
